# MPXV A29 — restrained Boltz-2 predictions, multi-seed

Runs one restraint set across several seeds in a single cell, then classifies the topology, verifies the restraints were satisfied, and summarises.

**Setup (Kaggle right-hand panel):** Accelerator → `GPU T4 x2` or `P100`; Internet → `On`. Both require a phone-verified account.

**Before closing:** run the packaging cell. The machine is wiped on disconnect.


---
## 1. Environment

In [1]:
import os
from pathlib import Path

WORK    = Path("/kaggle/working")
SCRATCH = Path("/kaggle/temp")
SCRATCH.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

ENV_DIR = SCRATCH / "boltz-env"
BOLTZ   = str(ENV_DIR / "bin" / "boltz")
PY      = str(ENV_DIR / "bin" / "python")
CACHE   = str(SCRATCH / "boltz_cache")
os.makedirs(CACHE, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
try:
    urllib.request.urlopen("https://pypi.org", timeout=5)
    print("Internet: ON")
except Exception:
    print("Internet: OFF - enable it in the right-hand panel.")

Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Internet: ON


---
## 2. Install Boltz-2 v2.2.1

In [2]:
!pip install -q uv
!uv venv {ENV_DIR} --python 3.12 --quiet
!uv pip install --python {PY} --quiet boltz==2.2.1 wrapt
!{PY} -c "import importlib.metadata as m; print('boltz', m.version('boltz'))"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 62.4 MB/s eta 0:00:00:00:0100:01
boltz 2.2.1


---
## 3. Target

In [ ]:
# ---- EDIT ---------------------------------------------------------------
TARGET_NAME = "MPXV_A29"
N_CHAINS    = 3
PRESET      = "Q77HM6"        # Q77HM6 | P0DTN3 | P11258 | 3VOP
# -------------------------------------------------------------------------

import re

PRESETS = {
 "Q77HM6": ("MPXV A29, clade I, 1-110", 1,
   "MDGTLFPGDDDLAIPATEFFSTKAAKNPETKREAIVKAYGDDNEETLKQRLTNLEKKITNITTKFEQIEKCCKHNDEVLFRLENHAETLRAAMISLAKKIDVQTGRRPYE"),
 "P0DTN3": ("MPXV A29, clade II, 1-110", 1,
   "MDGTLFPGDDDLAIPATEFFSTKAAKNPETKREAIVKAYGDDNEETLKQRLTNLEKKITNITTKFEQIEKCCKRNDEVLFRLENHAETLRAAMISLAKKIDVQTGRHPYE"),
 "P11258": ("VACV A27, 1-110", 1,
   "MDGTLFPGDDDLAIPATEFFSTKAAKKPEAKREAIVKADEDDNEETLKQRLTNLEKKITNVTTKFEQIEKCCKRNDEVLFRLENHAETLRAAMISLAKKIDVQTGRRPYE"),
 "3VOP":   ("VACV A27, 21-84, C71A/C72A", 21,
   "STKAAKKPEAKREAIVKADEDDNEETLKQRLTNLEKKITNVTTKFEQIEKAAKRNDEVLFRLEN"),
}

label, FIRST_RESIDUE, raw = PRESETS[PRESET]
SEQUENCE = re.sub(r"[^A-Za-z]", "", raw).upper()
LAST_RESIDUE = FIRST_RESIDUE + len(SEQUENCE) - 1

print(f"{TARGET_NAME}  |  {PRESET} - {label}")
print(f"{len(SEQUENCE)} aa, residues {FIRST_RESIDUE}-{LAST_RESIDUE}, homo-{N_CHAINS}mer")

---
## 4b. Template (optional)

An alternative to contact restraints: give Boltz the 3VOP biological assembly
as a structural template and let it steer the whole coiled coil rather than two
residue pairs.

The two approaches answer different questions and should be reported as separate
arms. **Restraints** impose only chain directionality at residues 47 and 65, so
the register, the second antiparallel interface and the packing emerge
unprompted and can serve as validation. **A template** transfers the register
and the geometry as well, so it cannot validate them — but it addresses the one
deficiency the restrained model has: the helices splay, giving 2.32 Å
whole-assembly RMSD to the crystal over residues 45–84 against 1.2–1.5 Å per
chain.

| Parameter | Meaning |
| --- | --- |
| `TEMPLATE` | `"none"`, `"3VOP"` (fetched from RCSB), or a path to your own CIF |
| `TEMPLATE_FORCE` | `False` passes the template as conditioning; `True` applies a steering potential |
| `TEMPLATE_THRESHOLD` | how far, in Å, the prediction may deviate from the template. `None` leaves it unset |
| `TEMPLATE_CHAINS` | which model chains the template applies to |

**Numbering.** 3VOP is deposited with its own numbering and covers only the
ordered region. Boltz matches templates by sequence, so it should align the
fragment to the corresponding part of your sequence without help — but check the
run log, and confirm afterwards that the coiled-coil region moved toward the
crystal rather than some other part of the chain.

**Peptide structures 5EOQ and 5EOR are deliberately not offered here.** Both are
short peptides crystallised in an antibody groove, so their conformations are
antibody-shaped; using them would impose a bound state on an apo model and make
any later agreement with the antibody literature circular. They also cover
residues 31–40 and 101–110, which is where the ensemble is most variable.

Templates and restraints can be combined, but if both are active you cannot say
which produced the result. Run them separately first.

In [ ]:
# ---- EDIT ---------------------------------------------------------------
TEMPLATE           = "none"      # "none" | "3VOP" | path to a .cif file
TEMPLATE_FORCE     = False       # True applies a steering potential
TEMPLATE_THRESHOLD = None        # deviation allowed, in angstroms; None = unset
TEMPLATE_CHAINS    = ["A", "B", "C"]   # model chains the template applies to
# -------------------------------------------------------------------------

from pathlib import Path

template_path = None
template_note = "none"

if TEMPLATE == "3VOP":
    template_path = WORK / "3vop-assembly1.cif"
    if not template_path.exists():
        print("Fetching 3VOP biological assembly from RCSB...")
        !curl -sL -o {template_path} https://files.rcsb.org/download/3vop-assembly1.cif
    if template_path.exists() and template_path.stat().st_size > 1000:
        template_note = "PDB 3VOP, biological assembly 1"
        print(f"Template: {template_path.name} "
              f"({template_path.stat().st_size/1000:.0f} kB)")
    else:
        raise SystemExit("Template download failed - check Internet is enabled.")

elif TEMPLATE != "none":
    template_path = Path(TEMPLATE).expanduser().resolve()
    if not template_path.is_file():
        raise SystemExit(f"Template not found: {template_path}")
    template_note = template_path.name
    print(f"Template: {template_path}")

else:
    print("No template (restraint-only arm)")

if template_path is not None:
    bits = ["tpl"]
    if TEMPLATE_THRESHOLD is not None:
        bits.append(f"t{TEMPLATE_THRESHOLD}")
    if TEMPLATE_FORCE:
        bits.append("F")
    TEMPLATE_TAG = "-" + "-".join(bits)
    template_note += (f"; force={TEMPLATE_FORCE}"
                      f"; threshold={TEMPLATE_THRESHOLD}"
                      f"; chains {','.join(TEMPLATE_CHAINS)}")
else:
    TEMPLATE_TAG = ""

print(f"Template tag: '{TEMPLATE_TAG}'")

---
## 4. Restraint set

Contacts from the 3VOP biological assembly, native A27/A29 numbering.

The screen found that **reciprocity is the requirement**: sets pinning residue 47 of one chain against 65 of another *and* the reverse (S1–S4) give the antiparallel arrangement; sets pinning one end only (S5, S6) or using mid-interface offsets (S8) do not, and their constraints are left violated.

Against the crystal, S1, S2 and S4 are indistinguishable (non-anchored chains 0.9–2.3 Å), so the minimal set is preferred.

Set `SEEDS` to the list you want. Each seed writes to its own folder.

In [ ]:
# ---- EDIT ---------------------------------------------------------------
SETS = {
    "S1": [("A",47,"C",65,6.33), ("A",65,"C",47,7.68)],
    "S2": [("A",47,"C",65,6.33), ("A",65,"C",47,7.68),
           ("B",47,"C",65,6.07), ("B",65,"C",47,7.75)],
    "S3": [("A",47,"B",47,7.54), ("A",65,"B",65,7.16),
           ("A",47,"C",65,6.33), ("A",65,"C",47,7.68)],
    "S4": [("A",47,"C",65,6.33), ("A",65,"C",47,7.68),
           ("B",47,"C",65,6.07), ("B",65,"C",47,7.75),
           ("A",47,"B",47,7.54), ("A",65,"B",65,7.16)],
    "S5": [("A",47,"C",65,6.33), ("B",47,"C",65,6.07)],
    "S6": [("A",65,"C",47,7.68), ("B",65,"C",47,7.75)],
    "S7": [("A",47,"C",65,6.33), ("A",65,"C",47,7.68),
           ("B",47,"C",65,6.07), ("B",65,"C",47,7.75),
           ("A",54,"C",57,5.88), ("B",51,"C",62,5.21)],
    "S8": [("A",54,"C",57,5.88), ("A",57,"C",54,6.00),
           ("B",51,"C",62,5.21), ("B",62,"C",51,5.68)],
}

SET_NAME = "S1"            # key from SETS, or "none" for the unrestrained control
DROP     = None            # 0-based index of one restraint to omit (ablation)
SEEDS    = [1]            #indicate seeds here separated by comma

MAX_DISTANCE = 8.0
FORCE        = False
SEPARATE_ENTITIES = True   # one entity per chain - needed for chain-specific restraints

DIFFUSION_SAMPLES = 5
RECYCLING_STEPS   = 3
SAMPLING_STEPS    = 200
STEP_SCALE        = 1.5
USE_MSA_SERVER    = True
USE_POTENTIALS    = True
# -------------------------------------------------------------------------

if SET_NAME == "none":
    active = []
else:
    active = list(SETS[SET_NAME])
    if DROP is not None:
        active = [r for i, r in enumerate(active) if i != DROP]

RUN_TAG_BASE = f"{SET_NAME}{f'-d{DROP}' if DROP is not None else ''}-{'F' if FORCE else 'f'}"
RUN_TAG = RUN_TAG_BASE + globals().get("TEMPLATE_TAG", "")

print(f"Variant {RUN_TAG}: {len(active)} restraints, force={FORCE}, seeds {SEEDS}")
for i, (c1, r1, c2, r2, d) in enumerate(active):
    print(f"  [{i}] {c1}/{r1} - {c2}/{r2}   crystal {d} A, bound {MAX_DISTANCE} A")

---
## 4b. Template (optional)

An alternative to contact restraints: give Boltz the 3VOP biological assembly
as a structural template and let it steer the whole coiled coil rather than two
residue pairs.

The two approaches answer different questions and should be reported as separate
arms. **Restraints** impose only chain directionality at residues 47 and 65, so
the register, the second antiparallel interface and the packing emerge
unprompted and can serve as validation. **A template** transfers the register
and the geometry as well, so it cannot validate them — but it addresses the one
deficiency the restrained model has: the helices splay, giving 2.32 Å
whole-assembly RMSD to the crystal over residues 45–84 against 1.2–1.5 Å per
chain.

| Parameter | Meaning |
| --- | --- |
| `TEMPLATE` | `"none"`, `"3VOP"` (fetched from RCSB), or a path to your own CIF |
| `TEMPLATE_FORCE` | `False` passes the template as conditioning; `True` applies a steering potential |
| `TEMPLATE_THRESHOLD` | how far, in Å, the prediction may deviate from the template. `None` leaves it unset |
| `TEMPLATE_CHAINS` | which model chains the template applies to |

**Numbering.** 3VOP is deposited with its own numbering and covers only the
ordered region. Boltz matches templates by sequence, so it should align the
fragment to the corresponding part of your sequence without help — but check the
run log, and confirm afterwards that the coiled-coil region moved toward the
crystal rather than some other part of the chain.

**Peptide structures 5EOQ and 5EOR are deliberately not offered here.** Both are
short peptides crystallised in an antibody groove, so their conformations are
antibody-shaped; using them would impose a bound state on an apo model and make
any later agreement with the antibody literature circular. They also cover
residues 31–40 and 101–110, which is where the ensemble is most variable.

Templates and restraints can be combined, but if both are active you cannot say
which produced the result. Run them separately first.

In [ ]:
# ---- EDIT ---------------------------------------------------------------
TEMPLATE           = "none"      # "none" | "3VOP" | path to a .cif file
TEMPLATE_FORCE     = False       # True applies a steering potential
TEMPLATE_THRESHOLD = None        # deviation allowed, in angstroms; None = unset
TEMPLATE_CHAINS    = ["A", "B", "C"]   # model chains the template applies to
# -------------------------------------------------------------------------

from pathlib import Path

template_path = None
template_note = "none"

if TEMPLATE == "3VOP":
    template_path = WORK / "3vop-assembly1.cif"
    if not template_path.exists():
        print("Fetching 3VOP biological assembly from RCSB...")
        !curl -sL -o {template_path} https://files.rcsb.org/download/3vop-assembly1.cif
    if template_path.exists() and template_path.stat().st_size > 1000:
        template_note = "PDB 3VOP, biological assembly 1"
        print(f"Template: {template_path.name} "
              f"({template_path.stat().st_size/1000:.0f} kB)")
    else:
        raise SystemExit("Template download failed - check Internet is enabled.")

elif TEMPLATE != "none":
    template_path = Path(TEMPLATE).expanduser().resolve()
    if not template_path.is_file():
        raise SystemExit(f"Template not found: {template_path}")
    template_note = template_path.name
    print(f"Template: {template_path}")

else:
    print("No template (restraint-only arm)")

if template_path is not None:
    bits = ["tpl"]
    if TEMPLATE_THRESHOLD is not None:
        bits.append(f"t{TEMPLATE_THRESHOLD}")
    if TEMPLATE_FORCE:
        bits.append("F")
    TEMPLATE_TAG = "-" + "-".join(bits)
    template_note += (f"; force={TEMPLATE_FORCE}"
                      f"; threshold={TEMPLATE_THRESHOLD}"
                      f"; chains {','.join(TEMPLATE_CHAINS)}")
else:
    TEMPLATE_TAG = ""

print(f"Template tag: '{TEMPLATE_TAG}'")

---
## 5. Input file

In [ ]:
import string

CHAIN_IDS = list(string.ascii_uppercase[:N_CHAINS])
yaml_text = "version: 1\nsequences:\n"

if SEPARATE_ENTITIES:
    for cid in CHAIN_IDS:
        yaml_text += f"  - protein:\n      id: {cid}\n      sequence: {SEQUENCE}\n"
else:
    yaml_text += ("  - protein:\n"
                  f"      id: [{', '.join(CHAIN_IDS)}]\n"
                  f"      sequence: {SEQUENCE}\n")

if active:
    yaml_text += "constraints:\n"
    for c1, r1, c2, r2, _ in active:
        yaml_text += ("  - contact:\n"
                      f"      token1: [{c1}, {r1}]\n"
                      f"      token2: [{c2}, {r2}]\n"
                      f"      max_distance: {MAX_DISTANCE}\n"
                      f"      force: {str(FORCE).lower()}\n")

if template_path is not None:
    yaml_text += "templates:\n"
    yaml_text += f"  - cif: {template_path}\n"
    if TEMPLATE_CHAINS:
        yaml_text += f"      chain_id: [{', '.join(TEMPLATE_CHAINS)}]\n"
    if TEMPLATE_FORCE:
        yaml_text += "      force: true\n"
    if TEMPLATE_THRESHOLD is not None:
        yaml_text += f"      threshold: {TEMPLATE_THRESHOLD}\n"

yaml_path = WORK / f"{TARGET_NAME}_{RUN_TAG}.yaml"
yaml_path.write_text(yaml_text)
print(yaml_text)

---
## 6. Run all seeds

One cell, one seed after another. The first seed downloads the weights and builds the MSA; later seeds reuse both, so they are considerably faster.

Roughly 10–15 min per seed for a 330-residue trimer, so five seeds is about an hour. Kaggle tolerates inactivity, but keep the tab open.

In [12]:
import time, datetime

t_all = time.time()
for SEED in SEEDS:
    out_dir = WORK / f"results_{TARGET_NAME}_{RUN_TAG}_seed{SEED}"
    if list(out_dir.rglob("*.cif")):
        print(f"seed {SEED}: already done, skipping")
        continue

    cmd = (f"{BOLTZ} predict {yaml_path}"
           f" --out_dir {out_dir}"
           f" --cache {CACHE}"
           f" --diffusion_samples {DIFFUSION_SAMPLES}"
           f" --recycling_steps {RECYCLING_STEPS}"
           f" --sampling_steps {SAMPLING_STEPS}"
           f" --step_scale {STEP_SCALE}"
           f" --seed {SEED}"
           f" --write_full_pae --override")
    if USE_MSA_SERVER:
        cmd += " --use_msa_server"
    if USE_POTENTIALS:
        cmd += " --use_potentials"

    print(f"\n=== seed {SEED} — started {datetime.datetime.now():%H:%M:%S} ===")
    t0 = time.time()
    !{cmd}
    n = len(list(out_dir.rglob("*.cif")))
    print(f"seed {SEED}: {n} models, {(time.time()-t0)/60:.1f} min")

print(f"\nAll seeds done in {(time.time()-t_all)/60:.1f} min")


=== seed 1 — started 11:21:39 ===
Seed set to 1
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
Processing 1 inputs with 1 threads.
  0%|                                                    | 0/1 [00:00<?, ?it/s]Generating MSA for /kaggle/working/MPXV_A29_S1-d1-f.yaml with 1 protein entities.
Calling MSA server for target MPXV_A29_S1-d1-f with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server

100%|████████████████████████████████████████████| 1/1 [00:00<00:00,  1.22it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/kaggle/temp/boltz-env/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch

---
## 7. Topology and restraint check

For every model: the chain-orientation class, and whether each imposed restraint was satisfied.

**P** = all three helix-axis dot products above +0.7 (parallel C3). **AP** = one above +0.7 and two below −0.7 (two parallel, one antiparallel). **X** = anything else.

A variant that comes out P *with restraints violated* means the model preferred a parallel arrangement over satisfying the constraints — not that the constraints failed to reach it.

In [13]:
!pip install -q gemmi
import gemmi, numpy as np, itertools, json

CC_START, CC_END = 44, 84

def axes(st):
    out = {}
    for ch in st[0]:
        pts = []
        for r in ch:
            n = r.seqid.num + FIRST_RESIDUE - 1
            if CC_START <= n <= CC_END:
                ca = r.find_atom("CA", "*")
                if ca: pts.append([ca.pos.x, ca.pos.y, ca.pos.z])
        if len(pts) < 8: continue
        P = np.array(pts)
        v = np.linalg.svd(P - P.mean(axis=0), full_matrices=False)[2][0]
        if np.dot(P[-1] - P[0], v) < 0: v = -v
        out[ch.name] = v / np.linalg.norm(v)
    return out

def cb(st, ch, num):
    for r in st[0][ch]:
        if r.seqid.num == num:
            a = r.find_atom("CB", "*") or r.find_atom("CA", "*")
            return a.pos
    return None

counts = {}
cifs = sorted(WORK.glob(f"results_{TARGET_NAME}_{RUN_TAG}_seed*/**/*.cif"))

for c in cifs:
    st = gemmi.read_structure(str(c)); st.setup_entities()
    ax = axes(st)
    dots = {f"{a}-{b}": float(np.dot(ax[a], ax[b]))
            for a, b in itertools.combinations(sorted(ax), 2)}
    v = list(dots.values())
    cls = "P" if all(x > 0.7 for x in v) else \
          "AP" if (sum(x > 0.7 for x in v) == 1 and sum(x < -0.7 for x in v) == 2) else "X"
    counts[cls] = counts.get(cls, 0) + 1

    viol = []
    for c1, r1, c2, r2, _ in active:
        p1, p2 = cb(st, c1, r1), cb(st, c2, r2)
        if p1 and p2 and p1.dist(p2) > MAX_DISTANCE:
            viol.append(f"{c1}{r1}-{c2}{r2}={p1.dist(p2):.1f}")

    flag = "" if not viol else "  VIOLATED: " + ", ".join(viol)
    print(f"{c.name:52s} {cls:3s}  " +
          "  ".join(f"{k}={x:+.2f}" for k, x in dots.items()) + flag)

print(f"\n{RUN_TAG}: {len(cifs)} models — {counts}")

MPXV_A29_S1-d1-f_model_0.cif                         P    A-B=+0.91  A-C=+0.91  B-C=+0.91  VIOLATED: A47-C65=26.5
MPXV_A29_S1-d1-f_model_1.cif                         P    A-B=+0.91  A-C=+0.91  B-C=+0.91  VIOLATED: A47-C65=26.2
MPXV_A29_S1-d1-f_model_2.cif                         P    A-B=+0.91  A-C=+0.91  B-C=+0.91  VIOLATED: A47-C65=26.4
MPXV_A29_S1-d1-f_model_3.cif                         P    A-B=+0.91  A-C=+0.91  B-C=+0.91  VIOLATED: A47-C65=26.3
MPXV_A29_S1-d1-f_model_4.cif                         P    A-B=+0.90  A-C=+0.90  B-C=+0.90  VIOLATED: A47-C65=26.3

S1-d1-f: 5 models — {'P': 5}


---
## 8. Parameters record

In [ ]:
import datetime, subprocess

ver = subprocess.run(
    f'{PY} -c "import importlib.metadata as m; print(m.version(\'boltz\'))"',
    shell=True, capture_output=True, text=True).stdout.strip()
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()

restr = "none" if not active else (
    f"{len(active)} contact (" +
    ", ".join(f"{c1}{r1}-{c2}{r2}" for c1, r1, c2, r2, _ in active) +
    f"); max_distance {MAX_DISTANCE} A; force={FORCE}")

print(f"""
Tool              : Boltz-2 (open source), v{ver}
Access            : Kaggle Notebooks, Python 3.12 environment via uv
Hardware          : {gpu}
Accessed          : {datetime.date.today().isoformat()}
Target            : {TARGET_NAME} ({PRESET} - {label})
Sequence          : {len(SEQUENCE)} aa, residues {FIRST_RESIDUE}-{LAST_RESIDUE}
Stoichiometry     : homo-{N_CHAINS}mer, {'one entity per chain' if SEPARATE_ENTITIES else 'single entity, three ids'}
Variant           : {RUN_TAG}
Templates         : {template_note}
Restraints        : {restr}
MSA               : {'automatic, ColabFold MMseqs2 server, greedy pairing' if USE_MSA_SERVER else 'none'}
Diffusion samples : {DIFFUSION_SAMPLES}
Recycling steps   : {RECYCLING_STEPS}
Sampling steps    : {SAMPLING_STEPS}
Step scale        : {STEP_SCALE}
Potentials        : {USE_POTENTIALS}
Seeds             : {', '.join(map(str, SEEDS))}
Models            : {len(cifs)}
Topology          : {counts}
""")

---
## 9. Package results

Two options: the current variant only, or everything in the session. Download from the **Output** panel on the right.

In [16]:
import shutil

ALL_VARIANTS = False   # True = every results_* folder in the session

pattern = "results_*" if ALL_VARIANTS else f"results_{TARGET_NAME}_{RUN_TAG}_seed*"
name    = f"{TARGET_NAME}_all" if ALL_VARIANTS else f"{TARGET_NAME}_{RUN_TAG}"

bundle = SCRATCH / "bundle"
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir(parents=True)

dirs = sorted(WORK.glob(pattern))
for d in dirs:
    shutil.copytree(d, bundle / d.name)
for y in WORK.glob("*.yaml"):
    shutil.copy(y, bundle)

zip_path = shutil.make_archive(str(WORK / name), "zip", str(bundle))
print(f"{len(dirs)} folders, {os.path.getsize(zip_path)/1e6:.1f} MB")
print(f"Written: {zip_path} — download from the Output panel.")

1 folders, 4.2 MB
Written: /kaggle/working/MPXV_A29_S1-d1-f.zip — download from the Output panel.


---
## Notes

### Running a different variant
Change `SET_NAME` (and `DROP` for the ablation) in section 4, then re-run sections 4–8. Every variant writes to `results_<target>_<RUN_TAG>_seed<n>` and all analysis cells filter by `RUN_TAG`, so variants never pool. Section 6 skips a seed that already has models, so re-running is safe.

### Ablation series
Fix `SET_NAME` and step `DROP` through 0 to *n*−1 at the same seeds. The proportion of models retaining the antiparallel arrangement after each removal identifies which restraints are load-bearing. For a two-restraint set, dropping either should collapse the topology.

### Chain declaration
`SEPARATE_ENTITIES = True` declares one entity per chain. With a single entity carrying three ids, the chains may be treated as symmetry-equivalent and chain-specific restraints have nothing to act on.

### Chain labelling in the output
Boltz output may place chains under segment identifiers, so `chain C` in PyMOL can select the wrong molecule or nothing at all. Check with `cmd.get_chains()` and, if needed, `alter <obj> and segi C, chain="C"` before any superposition. Confirm which chain is antiparallel from the helix-axis dot products rather than trusting the label.

### Reproducibility
Boltz is deterministic per seed only on identical hardware and software. Keep a whole comparison set on one platform and record which.


### Template arm
Set `TEMPLATE = "3VOP"` in section 4b. Output folders gain a `-tpl` suffix, so
template runs never pool with restraint-only runs.

Suggested series, all at the same seeds:

| Run | `SET_NAME` | `TEMPLATE` | `TEMPLATE_THRESHOLD` | Question |
| --- | --- | --- | --- | --- |
| 1 | `"S1"` | `"none"` | – | the restrained arm (already done) |
| 2 | `"none"` | `"3VOP"` | `None` | does the template alone give the antiparallel topology? |
| 3 | `"none"` | `"3VOP"` | `2.0` | does a tighter threshold reduce the splaying? |
| 4 | `"none"` | `"3VOP"` | `1.0` | tighter still — at what point does it distort? |

Run 2 is the informative one: if the template alone recovers the topology, that
is a second route to the same answer and worth reporting beside the restraints.
Measure each against the crystal with the anchored superposition and the
whole-assembly RMSD, and compare against 2.32 Å for the restrained model.

Do not combine a template with restraints in a reported run — with both active
you cannot say which produced the result.
